In [2]:
!pip install catboost

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold, GridSearchCV, RandomizedSearchCV

from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer, make_column_selector, make_column_transformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, VotingRegressor, StackingRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.5 MB/s eta 0:00:00


In [3]:
road = pd.read_csv("train.csv")
road

,id,road_type,num_lanes,curvature,speed_limit,lighting,weather,road_signs_present,public_road,time_of_day,holiday,school_season,num_reported_accidents,accident_risk
0,0,urban,2,0.06,35,daylight,rainy,False,True,afternoon,False,True,1,0.13
1,1,urban,4,0.99,35,daylight,clear,True,False,evening,True,True,0,0.35
2,2,rural,4,0.63,70,dim,clear,False,True,morning,True,False,2,0.30
3,3,highway,4,0.07,35,dim,rainy,True,True,morning,False,False,1,0.21
4,4,rural,1,0.58,60,daylight,foggy,False,False,evening,True,False,1,0.56
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
517749,517749,highway,4,0.10,70,daylight,foggy,True,True,afternoon,False,False,2,0.32
517750,517750,rural,4,0.47,35,daylight,rainy,True,True,morning,False,False,1,0.26
517751,517751,urban,4,0.62,25,daylight,foggy,False,False,afternoon,False,True,0,0.19
517752,517752,highway,3,0.63,25,night,clear,True,False,afternoon,True,True,3,0.51


In [5]:
ohe = OneHotEncoder(sparse_output = False, drop = 'first',handle_unknown = 'ignore').set_output(transform = 'pandas')
transformer = ColumnTransformer(transformers = [('ohe', ohe, make_column_selector(dtype_include = object))], remainder = 'passthrough',
                                verbose_feature_names_out = False).set_output(transform = 'pandas')

In [6]:
X, y = road.drop('accident_risk', axis = 1), road['accident_risk']

In [7]:
ss = StandardScaler()

In [8]:
lr = LinearRegression()
en = ElasticNet(random_state=26, max_iter=2000)
dtr = DecisionTreeRegressor(max_depth=10, random_state=26)
knn = KNeighborsRegressor(n_neighbors=5)
rfr = RandomForestRegressor(n_estimators=50, max_depth=12, random_state=26, n_jobs=-1, max_features='sqrt')

In [9]:
vote = VotingRegressor(estimators = [('LR', lr), ('EN', en), ('DTR', dtr), ('KNN', knn), ('RFR', rfr)])
pipe = Pipeline(steps = [('transformer', transformer), ('scaler', ss), ('voting', vote)])
pipe.get_params()

{'memory': None,
 'steps': [('transformer',
   ColumnTransformer(remainder='passthrough',
                     transformers=[('ohe',
                                    OneHotEncoder(drop='first',
                                                  handle_unknown='ignore',
                                                  sparse_output=False),
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x7a1e89ebec90>)],
                     verbose_feature_names_out=False)),
  ('scaler', StandardScaler()),
  ('voting',
   VotingRegressor(estimators=[('LR', LinearRegression()),
                               ('EN', ElasticNet(max_iter=2000, random_state=26)),
                               ('DTR',
                                DecisionTreeRegressor(max_depth=10,
                                                      random_state=26)),
                               ('KNN', KNeighborsRegressor()),
                               ('RFR',
        

In [10]:
from sklearn.model_selection import RandomizedSearchCV

kfolds = KFold(n_splits = 5, shuffle = True, random_state = 26)

params = {
    "voting__EN__alpha": [0.01, 0.1, 1.0, 5.0, 10.0],
    "voting__EN__l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9],
    "voting__DTR__max_depth": [8, 12, 15],
    "voting__DTR__min_samples_split": [2, 5],
    "voting__KNN__n_neighbors": [3, 5, 7],
    "voting__RFR__n_estimators": [50],
    "voting__RFR__max_depth": [10, 12]
}


gcv1 = RandomizedSearchCV(estimator=pipe, param_distributions=params, cv=kfolds,
                          n_iter=10, n_jobs=-1, verbose=1, random_state=26, scoring='r2')
gcv1.fit(X, y)
print("Best Params:", gcv1.best_params_)
print("Best R2 Score:", gcv1.best_score_)

Fitting 5 folds for each of 10 candidates, totalling 50 fits


KeyboardInterrupt: 

In [ ]:
# Time estimation helper
print(f"Dataset shape: {X.shape}")
print(f"n_samples: {X.shape[0]}, n_features: {X.shape[1]}")
print(f"\n--- TIME ESTIMATE ---")
print(f"Total fits: 30 iterations × 5 folds = 150 model fits")
print(f"Parallel jobs: 2 cores")
print(f"Expected time: 2-5 minutes on modern CPU (i5/Ryzen 5+)")
print(f"On slower machine: 5-15 minutes")
print(f"\nStarting RandomizedSearchCV...")

Dataset shape: (517754, 13)
n_samples: 517754, n_features: 13

--- TIME ESTIMATE ---
Total fits: 30 iterations × 5 folds = 150 model fits
Parallel jobs: 2 cores
Expected time: 2-5 minutes on modern CPU (i5/Ryzen 5+)
On slower machine: 5-15 minutes

Starting RandomizedSearchCV...


In [ ]:
# Evaluate best model
best_model = gcv1.best_estimator_
y_pred = best_model.predict(X)
print(f"R² Score: {r2_score(y, y_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y, y_pred)):.4f}")
print(f"MAE: {mean_absolute_error(y, y_pred):.4f}")

# For test data predictions later
gbr = GradientBoostingRegressor(random_state=26, n_estimators=50, max_depth=8)
xgbr = XGBRegressor(verbose=-1, random_state=26, n_estimators=50, max_depth=8, n_jobs=-1)
cbm = CatBoostRegressor(verbose=0, random_state=26, iterations=50)